# 🤖 LLM Fundamentals for Business Managers
### Essential Basics You Should Know (15-20 minutes)

---
## ⚙️ Setup - Run This First

In [4]:

# Install required packages
!pip install litellm tiktoken python-dotenv ipython -q

import os
from dotenv import load_dotenv
from litellm import completion
import tiktoken
from IPython.display import Markdown, display
load_dotenv()

print("✅ Setup complete. You are ready to go!")


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


✅ Setup complete. You are ready to go!


---
## 1. What Are LLMs Really Doing?

Large Language Models are **advanced next-word predictors**. 

They don't truly understand — they predict what word is most likely to come next based on patterns seen during training.

---
## 2. Tokens — The Real Unit of Cost

In [5]:
enc = tiktoken.encoding_for_model("gpt-4o")

examples = ["Hello, how are you?", "你好吗?", "Apa khabar?", "Hallo, wie geht's dir?"]

for text in examples:
    words = len(text.split())
    tokens = len(enc.encode(text))
    print(f"Words: {words} | Tokens: {tokens} | Ratio: {(tokens/words) if words > 0 else 0:.1f} tokens per word")

Words: 4 | Tokens: 6 | Ratio: 1.5 tokens per word
Words: 1 | Tokens: 3 | Ratio: 3.0 tokens per word
Words: 2 | Tokens: 4 | Ratio: 2.0 tokens per word
Words: 4 | Tokens: 7 | Ratio: 1.8 tokens per word


**Business Insight:** You are billed per token, not per word.

💡 **Cost Context:** Summarizing a 10-page document (~5,000 tokens) with gpt-4o-mini costs roughly $0.0005 less than a penny.

---
## 3. Next Word Prediction

In [6]:
def show_probability(prompt):
    resp = completion(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,
        logprobs=True,
        top_logprobs=5
    )
    print(f"Prompt: '{prompt}'")
    for item in resp.choices[0].logprobs.content[0].top_logprobs:
        prob = round((2.71828 ** item.logprob) * 100, 1)
        print(f"  {item.token:<15} {prob:6.1f}%")

show_probability("The capital of France is")
show_probability("A good way to reduce IT support costs is")
show_probability("A man was running so fast because")

Prompt: 'The capital of France is'
  Paris             99.9%
  The                0.1%
  the                0.0%
   Paris             0.0%
  Par                0.0%
Prompt: 'A good way to reduce IT support costs is'
  A                 46.0%
  to                30.6%
  There              7.7%
  Reduc              6.0%
  implement          5.5%
Prompt: 'A man was running so fast because'
  there             81.7%
  he                17.7%
  There              0.3%
  A                  0.1%
  The                0.1%


---
## Temperature & The Hallucination Warning

### Temperature: Predictable vs Creative

In [7]:

CREATIVE_PROMPT = "Write one short, catchy tagline for an AI-powered IT helpdesk. Max 10 words."

for label, temp in [("temp=0.0", 0.0), ("temp=1.0", 1.0)]:
    seen = set()
    print(f"   {label} (5 runs):")
    for i in range(5):
        r = completion(
            model="gpt-4o",
            messages=[{"role": "user", "content": CREATIVE_PROMPT}],
            temperature=temp, max_tokens=30
        )
        ans = r.choices[0].message.content.strip()
        seen.add(ans)
        print(f"     Run {i+1}: {ans}")
    print(f"   → Unique answers: {len(seen)}/5")
    print()

print("🔑 RULE: temp=0 for decisions/routing. temp=0.7–1.0 for drafting/brainstorming.")
print("   In any automation based pipeline: ALWAYS use temp=0.")



   temp=0.0 (5 runs):
     Run 1: "Smart Solutions, Instant Support: Your AI IT Helpdesk!"
     Run 2: "Smart Solutions, Instant Support: Your AI IT Helpdesk!"
     Run 3: "Smart Solutions, Instant Support: Your AI IT Helpdesk!"
     Run 4: "Smart Solutions, Instant Support: Your AI IT Helpdesk!"
     Run 5: "Smart Solutions, Instant Support: Your AI IT Helpdesk!"
   → Unique answers: 1/5

   temp=1.0 (5 runs):
     Run 1: "Smart Support, Swift Solutions: Your AI IT Helpdesk!"
     Run 2: "Swift Solutions: AI-Driven Helpdesk, Human-Perfected Service!"
     Run 3: "Swift Solutions, Powered by AI Precision!"
     Run 4: "Empowering Support, Accelerating Solutions: AI Helpdesk Revolution!"
     Run 5: "Smart Solutions, Instant Support: AI-Powered IT Helpdesk!"
   → Unique answers: 5/5

🔑 RULE: temp=0 for decisions/routing. temp=0.7–1.0 for drafting/brainstorming.
   In any automation based pipeline: ALWAYS use temp=0.


**Business Insight:**
- **Low Temperature (0.0-0.3):** More predictable, repeatable answers — best for factual tasks, legal, finance, compliance
- **High Temperature (0.7-1.0):** More creative, varied answers — best for brainstorming, marketing, content generation

---

### ⚠️ CRITICAL WARNING: Hallucination

LLMs are **not fact retrieval systems**. They can generate convincing but completely false information — this is called **hallucination**.

**Real Business Impact:**
- A legal summary might cite a non-existent court case
- A financial report might include made-up numbers
- A customer email might reference a policy that doesn't exist

**Rule:** Always verify important outputs. Human oversight is non-negotiable for business-critical decisions.

---
## 4. System Prompts — Research vs Legal

In [8]:
def ask(system, user_prompt, model="gpt-4o-mini"):
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0
    )
    return resp.choices[0].message.content.strip()

# Example 1: Research Style
research_system = "You are a professional researcher. Be factual, objective, and cite sources when possible."
print("🔬 Research Style:")
display(Markdown(ask(research_system, "What are current trends in AI adoption in banking?")))

🔬 Research Style:


As of October 2023, the adoption of artificial intelligence (AI) in the banking sector has been characterized by several key trends:

1. **Enhanced Customer Experience**: Banks are increasingly using AI to improve customer service through chatbots and virtual assistants. These tools can handle a variety of customer inquiries, provide personalized recommendations, and operate 24/7, leading to improved customer satisfaction and engagement. According to a report by McKinsey, AI-driven customer service solutions can reduce response times and operational costs significantly (McKinsey, 2022).

2. **Risk Management and Fraud Detection**: AI is being leveraged to enhance risk management practices and detect fraudulent activities. Machine learning algorithms can analyze transaction patterns in real-time to identify anomalies that may indicate fraud. A study by the World Economic Forum highlighted that AI can reduce false positives in fraud detection by up to 80% (World Economic Forum, 2023).

3. **Credit Scoring and Underwriting**: AI is transforming the credit scoring process by incorporating alternative data sources, such as social media activity and transaction history, to assess creditworthiness. This trend is particularly beneficial for underbanked populations who may lack traditional credit histories. A report from Deloitte noted that AI-driven credit scoring models can improve lending decisions and reduce default rates (Deloitte, 2023).

4. **Operational Efficiency**: Banks are adopting AI to streamline operations and reduce costs. Robotic Process Automation (RPA) combined with AI is being used to automate repetitive tasks, such as data entry and compliance checks. According to a report by Accenture, banks that implement AI-driven automation can achieve cost savings of up to 30% (Accenture, 2023).

5. **Personalized Financial Services**: AI is enabling banks to offer more personalized financial products and services. By analyzing customer data, banks can tailor offerings to individual needs, such as personalized investment advice or customized loan products. This trend is supported by research from Capgemini, which found that 75% of customers prefer personalized banking experiences (Capgemini, 2023).

6. **Regulatory Compliance**: AI is being utilized to help banks comply with regulatory requirements more efficiently. Natural Language Processing (NLP) is being used to analyze regulatory texts and ensure that banks adhere to compliance standards. A report by PwC indicated that AI can help reduce compliance costs by automating reporting and monitoring processes (PwC, 2023).

7. **Investment in AI Talent and Infrastructure**: As banks recognize the importance of AI, there is a growing investment in AI talent and infrastructure. Financial institutions are hiring data scientists and AI specialists to develop and implement AI strategies. Additionally, banks are investing in cloud computing and data analytics platforms to support AI initiatives.

These trends indicate a significant shift in how banks are leveraging AI technologies to enhance their operations, improve customer experiences, and manage risks more effectively. The ongoing evolution of AI in banking is expected to continue shaping the industry in the coming years.

### References
- McKinsey & Company. (2022). "The State of AI in Banking."
- World Economic Forum. (2023). "The Future of Financial Services: How AI is Transforming Banking."
- Deloitte. (2023). "AI in Banking: The Future of Credit Scoring."
- Accenture. (2023). "AI and Automation in Banking: A Path to Efficiency."
- Capgemini. (2023). "World Retail Banking Report."
- PwC. (2023). "AI in Financial Services: Compliance and Risk Management."

In [9]:
# Example 2: Legal / Compliance Style
legal_system = "You are a cautious legal advisor. Highlight risks clearly and never give definitive legal advice without proper disclaimer."
print("\n⚖️ Legal / Compliance Style:")
display(Markdown(ask(legal_system, "What are the risks of using AI to summarize customer contracts?")))


⚖️ Legal / Compliance Style:


Using AI to summarize customer contracts can present several risks that should be carefully considered:

1. **Inaccuracy**: AI models may misinterpret or overlook critical clauses, leading to summaries that do not accurately reflect the terms of the contract. This could result in misunderstandings or miscommunications.

2. **Loss of Context**: Contracts often contain nuanced language and context that AI may fail to capture. Important implications or obligations might be lost in a summary, potentially leading to legal disputes.

3. **Data Privacy Concerns**: If customer contracts contain sensitive personal or business information, using AI tools may raise data privacy issues, especially if the AI processes data in a way that is not compliant with regulations like GDPR or CCPA.

4. **Liability Issues**: If a summarized contract leads to a legal issue or dispute, there may be questions about liability. If the AI-generated summary is relied upon in a legal context, it could expose the user to legal risks.

5. **Regulatory Compliance**: Depending on the jurisdiction and the nature of the contracts, there may be specific legal requirements for how contracts must be summarized or presented. Failing to comply with these could lead to regulatory penalties.

6. **Dependence on Technology**: Relying heavily on AI for contract summaries may lead to a reduction in human oversight, which can be problematic if the AI fails to perform as expected.

7. **Intellectual Property Risks**: If the AI tool uses proprietary algorithms or data, there may be concerns about intellectual property rights, especially if the summaries are shared or used commercially.

8. **Bias and Fairness**: AI systems can inadvertently perpetuate biases present in the training data, which may lead to unfair or discriminatory outcomes in contract interpretation.

9. **Lack of Legal Expertise**: AI lacks the nuanced understanding of legal principles that a qualified attorney possesses. Relying solely on AI could lead to oversights that a legal professional would catch.

Given these risks, it is crucial to approach the use of AI for summarizing customer contracts with caution. It is advisable to have a qualified legal professional review any AI-generated summaries to ensure accuracy and compliance with applicable laws. 

**Disclaimer**: This response is for informational purposes only and does not constitute legal advice. For specific legal concerns, please consult a qualified attorney.

---
## 5. Same Question, Different Models + Cost Tracking

In [10]:
import os
import litellm
from litellm import completion
from dotenv import load_dotenv

# 1. Explicitly load environment variables from your .env file
load_dotenv()

# Global cost tracker
total_cost = 0.0

def compare_models_with_cost(prompt):
    global total_cost
    
    # 2. Use explicit 'provider/' prefixes and valid model names
    for model in ["gpt-4o", "anthropic/claude-haiku-4-5"]:
        try:
            # The completion function natively pulls ANTHROPIC_API_KEY 
            # and OPENAI_API_KEY from the environment variables we loaded.
            resp = completion(
                model=model, 
                messages=[{"role": "user", "content": prompt}], 
                temperature=0.0
            )
            
            # Extract token usage safely
            try:
                prompt_tokens = resp.usage.prompt_tokens
                completion_tokens = resp.usage.completion_tokens
                total_tokens = resp.usage.total_tokens
            except AttributeError:
                prompt_tokens = completion_tokens = total_tokens = 0
            
            # Calculate cost
            try:
                cost = litellm.completion_cost(resp)
                total_cost += cost
            except Exception:
                cost = 0.0
            
            print(f"\n🔹 {model}")
            print(f"   📝 Input tokens: {prompt_tokens}")
            print(f"   📝 Output tokens: {completion_tokens}")
            print(f"   📝 Total tokens: {total_tokens}")
            print(f"   💵 This call: ${cost:.6f}")
            print(f"   💰 Running total: ${total_cost:.6f}")
            print("   " + "-" * 50)
            display(Markdown(resp.choices[0].message.content.strip()))
            
        except Exception as e:
            print(f"\n❌ Error running model {model}: {e}")

compare_models_with_cost("Summarize the benefits and risks of using generative AI in our company.")


🔹 gpt-4o
   📝 Input tokens: 23
   📝 Output tokens: 456
   📝 Total tokens: 479
   💵 This call: $0.004618
   💰 Running total: $0.004618
   --------------------------------------------------


Using generative AI in your company can offer several benefits and risks. Here's a summary:

### Benefits:

1. **Increased Efficiency and Productivity:**
   - Automates repetitive tasks, freeing up employees to focus on more strategic activities.
   - Accelerates content creation, such as marketing materials, reports, and product descriptions.

2. **Enhanced Creativity and Innovation:**
   - Generates new ideas and solutions that may not be immediately apparent to human teams.
   - Facilitates rapid prototyping and experimentation in product development.

3. **Personalization and Customer Engagement:**
   - Creates personalized content and recommendations, improving customer experience and satisfaction.
   - Enhances customer interactions through AI-driven chatbots and virtual assistants.

4. **Cost Savings:**
   - Reduces the need for extensive human resources in certain areas, lowering operational costs.
   - Minimizes errors and inefficiencies, leading to cost-effective processes.

5. **Data Analysis and Insights:**
   - Analyzes large datasets to uncover patterns and insights that inform business decisions.
   - Supports predictive analytics for better forecasting and planning.

### Risks:

1. **Quality and Accuracy Concerns:**
   - May produce content that is inaccurate or lacks the nuance of human-generated work.
   - Requires oversight to ensure outputs meet company standards and values.

2. **Ethical and Bias Issues:**
   - Can perpetuate or amplify existing biases present in training data.
   - Raises ethical concerns regarding the use of AI-generated content and decision-making.

3. **Security and Privacy Risks:**
   - Potential for data breaches or misuse of sensitive information.
   - Requires robust security measures to protect proprietary and customer data.

4. **Dependence and Skill Gaps:**
   - Over-reliance on AI tools may lead to skill erosion among employees.
   - Necessitates ongoing training and upskilling to effectively integrate AI into workflows.

5. **Regulatory and Compliance Challenges:**
   - Must navigate evolving regulations and compliance requirements related to AI use.
   - Potential legal implications if AI-generated content infringes on intellectual property rights.

Balancing these benefits and risks involves careful planning, implementation, and monitoring to ensure that generative AI aligns with your company's goals and values while mitigating potential downsides.


🔹 anthropic/claude-haiku-4-5
   📝 Input tokens: 23
   📝 Output tokens: 318
   📝 Total tokens: 341
   💵 This call: $0.001613
   💰 Running total: $0.006231
   --------------------------------------------------


# Generative AI: Benefits & Risks

## Key Benefits

**Efficiency & Productivity**
- Automate routine tasks (writing, coding, analysis)
- Faster content creation and prototyping
- Reduced time on repetitive work

**Cost Savings**
- Lower labor costs for certain functions
- Faster project completion
- Reduced need for external contractors

**Innovation**
- New product/service possibilities
- Enhanced decision-making with data analysis
- Creative problem-solving support

**Scalability**
- Handle increased volume without proportional headcount growth
- 24/7 availability

## Key Risks

**Quality & Accuracy**
- Hallucinations and false information
- Inconsistent output requiring human review
- Potential errors in critical decisions

**Security & Privacy**
- Data exposure if sensitive info enters AI systems
- Intellectual property concerns
- Compliance issues (GDPR, industry regulations)

**Operational**
- Over-reliance on AI reducing human skills
- Integration challenges with existing systems
- Vendor lock-in risks

**Reputational & Legal**
- Bias in outputs affecting customers/employees
- Liability for AI-generated errors
- Unclear accountability

## Recommendation

Start with **low-risk pilots** in specific departments, establish clear governance policies, and maintain human oversight—especially for high-stakes decisions.

What specific use cases are you considering?

**💡 Cost Tracking Note:** LiteLLM provides **estimated** costs based on token counts. For exact billing, check your provider dashboard (OpenAI, Anthropic, etc.).

---
## Final Takeaways

1. LLMs predict likely words, not search facts
2. **Tokens** are your real cost unit — not words
3. **Temperature** controls predictability vs creativity
4. ⚠️ **Hallucination** is real — always verify critical outputs
5. **System Prompts** are very powerful — use them to set role and tone
6. Different models give different answers at different costs
7. Always consider cost, accuracy, and risk
8. Good prompting + human oversight = best results